# Trabajo 3: Análisis de datos con NumPy y Pandas

## Análisis de la red de tiendas RetailNow

En este notebook se analizan los datos de ventas, inventario y satisfacción del cliente de RetailNow.

El objetivo es utilizar **Pandas** para cargar, limpiar, agrupar y filtrar los datos, y **NumPy** para realizar cálculos estadísticos y una simulación sencilla de ventas futuras.

## 1. Importar las librerías

In [1]:
import pandas as pd
import numpy as np

## 2. Cargar y limpiar los datos

Las rutas se definen de forma absoluta, tal como se solicita en el enunciado.

Después de cargar cada CSV se eliminan las filas que contengan valores nulos mediante `dropna()`.

In [2]:
RUTA_VENTAS = "/workspace/sales.csv"
RUTA_INVENTARIOS = "/workspace/inventories.csv"
RUTA_SATISFACCION = "/workspace/satisfaction.csv"

try:
    ventas = pd.read_csv(RUTA_VENTAS)
    inventarios = pd.read_csv(RUTA_INVENTARIOS)
    satisfaccion = pd.read_csv(RUTA_SATISFACCION)
except FileNotFoundError as error:
    print("No se ha encontrado uno de los archivos CSV.")
    print("Comprueba que sales.csv, inventories.csv y satisfaction.csv estén en /workspace.")
    raise error

# Eliminamos las filas con valores nulos
ventas = ventas.dropna().copy()
inventarios = inventarios.dropna().copy()
satisfaccion = satisfaccion.dropna().copy()

print("Datos cargados y limpiados correctamente.")

Datos cargados y limpiados correctamente.


### Comprobación de la estructura de los DataFrames

In [3]:
print("VENTAS")
display(ventas.head())
ventas.info()

print("\nINVENTARIOS")
display(inventarios.head())
inventarios.info()

print("\nSATISFACCIÓN")
display(satisfaccion.head())
satisfaccion.info()

VENTAS


,ID_Tienda,Producto,Cantidad_Vendida,Precio_Unitario,Fecha_Venta
0,1,Producto A,20,100,2023-01-05
1,1,Producto B,15,200,2023-01-06
2,2,Producto A,30,100,2023-01-07
3,2,Producto C,25,300,2023-01-08
4,3,Producto A,10,100,2023-01-09


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   ID_Tienda         10 non-null     int64 
 1   Producto          10 non-null     object
 2   Cantidad_Vendida  10 non-null     int64 
 3   Precio_Unitario   10 non-null     int64 
 4   Fecha_Venta       10 non-null     object
dtypes: int64(3), object(2)
memory usage: 532.0+ bytes

INVENTARIOS


,ID_Tienda,Producto,Stock_Disponible,Fecha_Actualización
0,1,Producto A,50,2023-01-05
1,1,Producto B,40,2023-01-06
2,2,Producto A,60,2023-01-07
3,2,Producto C,45,2023-01-08
4,3,Producto A,30,2023-01-09


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   ID_Tienda            10 non-null     int64 
 1   Producto             10 non-null     object
 2   Stock_Disponible     10 non-null     int64 
 3   Fecha_Actualización  10 non-null     object
dtypes: int64(2), object(2)
memory usage: 452.0+ bytes

SATISFACCIÓN


,ID_Tienda,Satisfacción_Promedio,Fecha_Evaluación
0,1,85,2023-01-15
1,2,90,2023-01-15
2,3,70,2023-01-15
3,4,65,2023-01-15
4,5,55,2023-01-15


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 3 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   ID_Tienda              5 non-null      int64 
 1   Satisfacción_Promedio  5 non-null      int64 
 2   Fecha_Evaluación       5 non-null      object
dtypes: int64(2), object(1)
memory usage: 252.0+ bytes


## 3. Exploración de las ventas

Primero calculamos las ventas totales por producto y por tienda. En este caso, una venta representa la cantidad de unidades vendidas.

In [4]:
ventas_por_producto = (
    ventas.groupby("Producto", as_index=False)["Cantidad_Vendida"]
    .sum()
    .rename(columns={"Cantidad_Vendida": "Total_Ventas"})
)

ventas_por_tienda = (
    ventas.groupby("ID_Tienda", as_index=False)["Cantidad_Vendida"]
    .sum()
    .rename(columns={"Cantidad_Vendida": "Total_Ventas"})
)

print("Ventas totales por producto:")
display(ventas_por_producto)

print("Ventas totales por tienda:")
display(ventas_por_tienda)

Ventas totales por producto:


,Producto,Total_Ventas
0,Producto A,85
1,Producto B,75
2,Producto C,90


Ventas totales por tienda:


,ID_Tienda,Total_Ventas
0,1,35
1,2,55
2,3,50
3,4,60
4,5,50


### Ingresos totales por tienda

In [5]:
# Ingreso de cada registro de venta
ventas["Ingresos"] = ventas["Cantidad_Vendida"] * ventas["Precio_Unitario"]

ingresos_por_tienda = (
    ventas.groupby("ID_Tienda", as_index=False)["Ingresos"]
    .sum()
    .rename(columns={"Ingresos": "Ingresos_Totales"})
)

display(ingresos_por_tienda)

,ID_Tienda,Ingresos_Totales
0,1,5000
1,2,10500
2,3,9000
3,4,13000
4,5,13000


### Resumen estadístico

`describe()` permite obtener de forma rápida medidas como la media, desviación estándar, mínimo, máximo y cuartiles.  
La mediana corresponde al valor mostrado en la fila `50%`.

In [6]:
ventas.describe()

,ID_Tienda,Cantidad_Vendida,Precio_Unitario,Ingresos
count,10.000000,10.000000,10.000000,10.000000
mean,3.000000,25.000000,190.000000,5050.000000
std,1.490712,9.128709,87.559504,3361.960407
min,1.000000,10.000000,100.000000,1000.000000
25%,2.000000,20.000000,100.000000,2625.000000
50%,3.000000,25.000000,200.000000,3500.000000
75%,4.000000,30.000000,275.000000,7875.000000
max,5.000000,40.000000,300.000000,10500.000000


### Agrupación con `groupby()`

Los archivos proporcionados **no contienen una columna de categoría de producto**, por lo que no es posible calcular el promedio por tienda y categoría sin inventar información.

Sí podemos utilizar `groupby()` para obtener el promedio de unidades vendidas por tienda.

In [7]:
promedio_ventas_por_tienda = (
    ventas.groupby("ID_Tienda", as_index=False)["Cantidad_Vendida"]
    .mean()
    .rename(columns={"Cantidad_Vendida": "Promedio_Ventas"})
)

display(promedio_ventas_por_tienda)

,ID_Tienda,Promedio_Ventas
0,1,17.5
1,2,27.5
2,3,25.0
3,4,30.0
4,5,25.0


## 4. Análisis de inventarios

Para relacionar las ventas con el inventario se unen ambos DataFrames mediante `ID_Tienda` y `Producto`.

La rotación utilizada en este ejercicio se calcula como:

\[
\text{Rotación de inventario} =
\frac{\text{Cantidad vendida}}{\text{Stock disponible}}
\]

Después se calcula también la rotación total de cada tienda.

In [8]:
inventario_analisis = inventarios.merge(
    ventas[["ID_Tienda", "Producto", "Cantidad_Vendida"]],
    on=["ID_Tienda", "Producto"],
    how="left"
)

# Si un producto no tuviera ventas, consideramos que se vendieron 0 unidades.
inventario_analisis["Cantidad_Vendida"] = (
    inventario_analisis["Cantidad_Vendida"].fillna(0)
)

# Evitamos divisiones entre cero.
inventario_analisis["Rotacion_Inventario"] = np.where(
    inventario_analisis["Stock_Disponible"] > 0,
    inventario_analisis["Cantidad_Vendida"] / inventario_analisis["Stock_Disponible"],
    np.nan
)

display(inventario_analisis)

,ID_Tienda,Producto,Stock_Disponible,Fecha_Actualización,Cantidad_Vendida,Rotacion_Inventario
0,1,Producto A,50,2023-01-05,20,0.400000
1,1,Producto B,40,2023-01-06,15,0.375000
2,2,Producto A,60,2023-01-07,30,0.500000
3,2,Producto C,45,2023-01-08,25,0.555556
4,3,Producto A,30,2023-01-09,10,0.333333
5,3,Producto B,80,2023-01-10,40,0.500000
6,4,Producto C,70,2023-01-11,35,0.500000
7,4,Producto A,50,2023-01-12,25,0.500000
8,5,Producto B,40,2023-01-13,20,0.500000
9,5,Producto C,60,2023-01-14,30,0.500000


### Rotación de inventarios por tienda

In [9]:
rotacion_por_tienda = (
    inventario_analisis.groupby("ID_Tienda", as_index=False)
    .agg(
        Ventas_Totales=("Cantidad_Vendida", "sum"),
        Stock_Total=("Stock_Disponible", "sum")
    )
)

rotacion_por_tienda["Rotacion_Inventario"] = np.where(
    rotacion_por_tienda["Stock_Total"] > 0,
    rotacion_por_tienda["Ventas_Totales"] / rotacion_por_tienda["Stock_Total"],
    np.nan
)

display(rotacion_por_tienda)

,ID_Tienda,Ventas_Totales,Stock_Total,Rotacion_Inventario
0,1,35,90,0.388889
1,2,55,105,0.523810
2,3,50,110,0.454545
3,4,60,120,0.500000
4,5,50,100,0.500000


### Tiendas con inventario crítico

Según el enunciado, una tienda se considera crítica cuando sus ventas representan **menos del 10% del stock disponible**, es decir, cuando su rotación es inferior a `0.10`.

In [10]:
tiendas_inventario_critico = rotacion_por_tienda[
    rotacion_por_tienda["Rotacion_Inventario"] < 0.10
]

if tiendas_inventario_critico.empty:
    print("No hay tiendas con niveles críticos de inventario.")
else:
    display(tiendas_inventario_critico)

No hay tiendas con niveles críticos de inventario.


## 5. Satisfacción del cliente y relación con las ventas

Unimos la satisfacción media de cada tienda con sus ventas e ingresos para poder comparar ambos tipos de información.

In [11]:
rendimiento_tiendas = (
    satisfaccion
    .merge(ventas_por_tienda, on="ID_Tienda", how="left")
    .merge(ingresos_por_tienda, on="ID_Tienda", how="left")
)

display(rendimiento_tiendas)

,ID_Tienda,Satisfacción_Promedio,Fecha_Evaluación,Total_Ventas,Ingresos_Totales
0,1,85,2023-01-15,35,5000
1,2,90,2023-01-15,55,10500
2,3,70,2023-01-15,50,9000
3,4,65,2023-01-15,60,13000
4,5,55,2023-01-15,50,13000


### Tiendas con satisfacción inferior al 60%

In [12]:
tiendas_baja_satisfaccion = rendimiento_tiendas[
    rendimiento_tiendas["Satisfacción_Promedio"] < 60
]

display(tiendas_baja_satisfaccion)

if tiendas_baja_satisfaccion.empty:
    print("No hay tiendas con satisfacción inferior al 60%.")
else:
    print(
        "Recomendación: revisar las causas de las valoraciones bajas, "
        "especialmente la atención al cliente, la disponibilidad de productos "
        "y los tiempos de espera. Después conviene repetir la medición para "
        "comprobar si las medidas aplicadas mejoran la satisfacción."
    )

,ID_Tienda,Satisfacción_Promedio,Fecha_Evaluación,Total_Ventas,Ingresos_Totales
4,5,55,2023-01-15,50,13000


Recomendación: revisar las causas de las valoraciones bajas, especialmente la atención al cliente, la disponibilidad de productos y los tiempos de espera. Después conviene repetir la medición para comprobar si las medidas aplicadas mejoran la satisfacción.


## 6. Cálculos estadísticos con NumPy

Convertimos la columna `Total_Ventas` del DataFrame agrupado por tienda a un array de NumPy mediante `to_numpy()`.

Después calculamos la mediana y la desviación estándar utilizando exclusivamente funciones de **NumPy**, tal como solicita la actividad.

In [13]:
array_ventas = ventas_por_tienda["Total_Ventas"].to_numpy()

mediana_ventas = np.median(array_ventas)
desviacion_ventas = np.std(array_ventas)

print("Array de ventas:", array_ventas)
print("Mediana de las ventas totales:", mediana_ventas)
print("Desviación estándar de las ventas:", round(desviacion_ventas, 2))

Array de ventas: [35 55 50 60 50]
Mediana de las ventas totales: 50.0
Desviación estándar de las ventas: 8.37


## 7. Simulación de ventas futuras con NumPy

Para realizar una simulación sencilla se generan factores aleatorios alrededor de `1.05`.

Esto representa un escenario en el que las ventas futuras se sitúan, de media, aproximadamente un **5% por encima** de las ventas actuales, con cierta variabilidad.

Se establece `np.random.seed(42)` para que los resultados sean reproducibles.

In [14]:
np.random.seed(42)

numero_periodos = 6

factores_aleatorios = np.random.normal(
    loc=1.05,
    scale=0.10,
    size=(len(array_ventas), numero_periodos)
)

proyecciones = np.round(
    array_ventas[:, np.newaxis] * factores_aleatorios,
    2
)

# Evitamos valores negativos en una posible simulación extrema.
proyecciones = np.maximum(proyecciones, 0)

columnas = [f"Periodo_{i}" for i in range(1, numero_periodos + 1)]

proyecciones_df = pd.DataFrame(
    proyecciones,
    index=ventas_por_tienda["ID_Tienda"],
    columns=columnas
)

proyecciones_df.index.name = "ID_Tienda"

display(proyecciones_df)

,Periodo_1,Periodo_2,Periodo_3,Periodo_4,Periodo_5,Periodo_6
ID_Tienda,,,,,,
1,38.49,36.27,39.02,42.08,35.93,35.93
2,66.44,61.97,55.17,60.73,55.20,55.19
3,53.71,42.93,43.88,49.69,47.44,54.07
4,57.55,54.53,71.79,61.65,63.41,54.45
5,49.78,53.05,46.75,54.38,49.50,51.04


### Estadísticas básicas de las proyecciones

In [15]:
print("Media de las proyecciones:", round(np.mean(proyecciones), 2))
print("Mediana de las proyecciones:", round(np.median(proyecciones), 2))
print("Desviación estándar de las proyecciones:", round(np.std(proyecciones), 2))

Media de las proyecciones: 51.4
Mediana de las proyecciones: 53.38
Desviación estándar de las proyecciones: 9.13


## 8. Conclusiones

- La tienda con mayor cantidad de unidades vendidas es la **tienda 4**, con 60 unidades.
- Las tiendas 4 y 5 presentan los mayores ingresos totales, con 13.000 cada una.
- No se detectan tiendas cuya rotación total de inventario sea inferior al 10%.
- La **tienda 5** presenta una satisfacción media del **55%**, por debajo del límite del 60%.
- La mediana de las ventas totales por tienda es de **50 unidades**.
- La simulación permite obtener posibles escenarios de ventas futuras de forma reproducible gracias al uso de una semilla de NumPy.

Estos resultados permiten identificar rápidamente las tiendas que necesitan más atención y sirven como base para tomar decisiones sobre inventario, ventas y experiencia del cliente.